In [2]:
from pathlib import Path
import os
import random
import json
import pickle
from collections import Counter

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

/home/jamyoung/miniconda3/envs/yolov3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ROOT = Path("../preprocess/preprocessing")
META_PATH = ROOT / "metadata.csv"

EFF_CLS_DIR = Path("outputs_effnetv2_m_finetune_classifier")
EFF_CLS_DIR.mkdir(parents=True, exist_ok=True)

classes = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}

seed = 42
model_name = "tf_efficientnetv2_m"

image_size = 300
batch_size = 8
num_workers = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(seed)

print("device:", device)
print("model_name:", model_name)
print("root:", ROOT)
print("metadata:", META_PATH)

device: cuda
model_name: tf_efficientnetv2_m
root: ../preprocess/preprocessing
metadata: ../preprocess/preprocessing/metadata.csv


In [4]:
df = pd.read_csv(META_PATH)

print(df.head())
print(df.columns)
print(df.shape)
print(df["dx"].value_counts())

required_cols = [
    "lesion_id",
    "image_id",
    "dx",
    "dx_type",
    "age",
    "sex",
    "localization",
    "dataset"
]

for col in required_cols:
    assert col in df.columns, f"Missing column: {col}"

def get_image_path(row):
    cls = row["dx"]
    image_id = row["image_id"]
    path = ROOT / cls / "enhanced" / f"{image_id}.jpg"
    if path.exists():
        return str(path)
    return None

df["image_path"] = df.apply(get_image_path, axis=1)

missing = df["image_path"].isna().sum()
print("missing images:", missing)

if missing > 0:
    df[df["image_path"].isna()][["image_id", "dx", "image_path"]].to_csv(
        EFF_CLS_DIR / "missing_images.csv",
        index=False
    )
    raise FileNotFoundError(f"{missing} images missing. Check missing_images.csv")

df["label"] = df["dx"].map(label2idx).astype(int)

print(df[["image_id", "dx", "label", "image_path"]].head())
print(df["label"].value_counts().sort_index())

     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

        dataset  
0  vidir_modern  
1  vidir_modern  
2  vidir_modern  
3  vidir_modern  
4  vidir_modern  
Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'dataset'],
      dtype='object')
(10015, 8)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
missing images: 0
       image_id   dx  label                                         image_path
0  ISIC_0027419  bkl      2  ../preprocess/preprocessing/bkl/enhanced/ISIC_...
1  ISIC_0025030  bkl      2  ../prepr

In [5]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["dx"],
    random_state=seed
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train:", len(train_df))
print(train_df["dx"].value_counts())

print("\nval:", len(val_df))
print(val_df["dx"].value_counts())

train_df.to_csv(EFF_CLS_DIR / "train_split.csv", index=False)
val_df.to_csv(EFF_CLS_DIR / "val_split.csv", index=False)

train: 9013
dx
nv       6034
mel      1002
bkl       989
bcc       463
akiec     294
vasc      128
df        103
Name: count, dtype: int64

val: 1002
dx
nv       671
mel      111
bkl      110
bcc       51
akiec     33
vasc      14
df        12
Name: count, dtype: int64


In [6]:
train_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.03
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
class HAMImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(int(row["label"]), dtype=torch.long)
        image_id = row["image_id"]

        return image, label, image_id

In [8]:
train_dataset_cls = HAMImageDataset(train_df, transform=train_tfms)
val_dataset_cls = HAMImageDataset(val_df, transform=val_tfms)

train_loader_cls = DataLoader(
    train_dataset_cls,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

val_loader_cls = DataLoader(
    val_dataset_cls,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

batch = next(iter(train_loader_cls))
images, labels, image_ids = batch

print(images.shape)
print(labels.shape)
print(image_ids[:3])
print(labels[:10])

torch.Size([8, 3, 300, 300])
torch.Size([8])
['ISIC_0028345', 'ISIC_0031165', 'ISIC_0028873']
tensor([5, 5, 5, 1, 5, 4, 5, 5])


In [9]:
class_counts = train_df["label"].value_counts().sort_index().values
N = class_counts.sum()
C = len(classes)

class_weights = N / (C * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

for cls, count, weight in zip(classes, class_counts, class_weights):
    print(f"{cls:6s} count={count:5d}, weight={weight.item():.4f}")

akiec  count=  294, weight=4.3795
bcc    count=  463, weight=2.7809
bkl    count=  989, weight=1.3019
df     count=  103, weight=12.5007
mel    count= 1002, weight=1.2850
nv     count= 6034, weight=0.2134
vasc   count=  128, weight=10.0592


In [10]:
class EfficientNetV2Classifier(nn.Module):
    def __init__(self, model_name="tf_efficientnetv2_m", num_classes=7, pretrained=True):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg"
        )

        feat_dim = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(feat_dim, num_classes)
        )

    def forward(self, x):
        feat = self.backbone(x)
        logits = self.classifier(feat)
        return logits

In [11]:
train_dataset_cls = HAMImageDataset(train_df, transform=train_tfms)
val_dataset_cls = HAMImageDataset(val_df, transform=val_tfms)

train_loader_cls = DataLoader(
    train_dataset_cls,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

val_loader_cls = DataLoader(
    val_dataset_cls,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

In [13]:
eff_cls_model = EfficientNetV2Classifier(
    model_name=model_name,
    num_classes=len(classes),
    pretrained=True
).to(device)

criterion_cls = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer_cls = torch.optim.AdamW(
    eff_cls_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler_cls = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_cls,
    mode="max",
    factor=0.5,
    patience=3
)

In [14]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def train_one_epoch_cls(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels, _ in tqdm(loader, desc="Training EfficientNetV2"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = avg_loss

    return metrics


@torch.no_grad()
def evaluate_cls(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    all_image_ids = []

    for images, labels, image_ids in tqdm(loader, desc="Validating EfficientNetV2"):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        total_loss += loss.item() * images.size(0)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())
        all_image_ids.extend(list(image_ids))

    avg_loss = total_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = avg_loss

    return metrics, np.array(all_labels), np.array(all_preds), np.array(all_probs), all_image_ids

In [15]:
train_metrics = train_one_epoch_cls(
    eff_cls_model,
    train_loader_cls,
    optimizer_cls,
    criterion_cls,
    device
)

val_metrics, y_val_cls, y_pred_cls, y_prob_cls, val_image_ids_cls = evaluate_cls(
    eff_cls_model,
    val_loader_cls,
    criterion_cls,
    device
)

print("train:", train_metrics)
print("val:", val_metrics)
print("pred distribution:", Counter(y_pred_cls))

Validating EfficientNetV2: 100%|██████████| 126/126 [00:27<00:00,  4.59it/s]

train: {'accuracy': 0.6327526905580828, 'balanced_accuracy': np.float64(0.559640713558637), 'precision_macro': 0.4104619015702342, 'recall_macro': 0.559640713558637, 'f1_macro': 0.457353676735524, 'loss': 1.1685189728822585}
val: {'accuracy': 0.7524950099800399, 'balanced_accuracy': np.float64(0.7518140151698492), 'precision_macro': 0.5458144954013824, 'recall_macro': 0.7518140151698492, 'f1_macro': 0.6023746323106918, 'loss': 0.8339342664055407}
pred distribution: Counter({np.int64(5): 572, np.int64(2): 160, np.int64(1): 84, np.int64(4): 67, np.int64(0): 64, np.int64(6): 37, np.int64(3): 18})


In [16]:
EFF_CLS_DIR = Path("outputs_effnetv2_m_finetune_classifier")
EFF_CLS_DIR.mkdir(parents=True, exist_ok=True)

num_epochs = 5
patience = 7
best_f1 = -1
no_improve = 0
history = []

for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch}/{num_epochs}")

    train_metrics = train_one_epoch_cls(
        eff_cls_model,
        train_loader_cls,
        optimizer_cls,
        criterion_cls,
        device
    )

    val_metrics, y_val_cls, y_pred_cls, y_prob_cls, val_image_ids_cls = evaluate_cls(
        eff_cls_model,
        val_loader_cls,
        criterion_cls,
        device
    )

    scheduler_cls.step(val_metrics["f1_macro"])

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "lr": optimizer_cls.param_groups[0]["lr"]
    }

    history.append(row)
    print(row)
    print("pred distribution:", Counter(y_pred_cls))

    pd.DataFrame(history).to_csv(EFF_CLS_DIR / "training_history.csv", index=False)

    current_f1 = val_metrics["f1_macro"]

    if current_f1 > best_f1:
        best_f1 = current_f1
        no_improve = 0

        torch.save(
            {
                "model_state_dict": eff_cls_model.state_dict(),
                "backbone_state_dict": eff_cls_model.backbone.state_dict(),
                "classes": classes,
                "label2idx": label2idx,
                "idx2label": idx2label,
                "model_name": model_name,
                "image_size": image_size,
            },
            EFF_CLS_DIR / "best_effnetv2_classifier.pth"
        )

        pd.DataFrame([val_metrics]).to_csv(EFF_CLS_DIR / "metrics.csv", index=False)

        pred_df = pd.DataFrame({
            "image_id": val_image_ids_cls,
            "true_label": [idx2label[i] for i in y_val_cls],
            "pred_label": [idx2label[i] for i in y_pred_cls],
            "correct": y_val_cls == y_pred_cls
        })

        for i, cls in idx2label.items():
            pred_df[f"prob_{cls}"] = y_prob_cls[:, i]

        pred_df.to_csv(EFF_CLS_DIR / "predictions.csv", index=False)

        print(f"Saved best EfficientNetV2 classifier. val_f1_macro={best_f1:.4f}")

    else:
        no_improve += 1
        print(f"No improvement: {no_improve}/{patience}")

    if no_improve >= patience:
        print("Early stopping triggered.")
        break


Epoch 1/5


Validating EfficientNetV2: 100%|██████████| 126/126 [00:25<00:00,  4.98it/s]


{'epoch': 1, 'train_accuracy': 0.725507600133141, 'train_balanced_accuracy': np.float64(0.7033958178054716), 'train_precision_macro': 0.5478397081019635, 'train_recall_macro': 0.7033958178054716, 'train_f1_macro': 0.6061233210898305, 'train_loss': 0.8439610453063582, 'val_accuracy': 0.7495009980039921, 'val_balanced_accuracy': np.float64(0.7990187527864894), 'val_precision_macro': 0.533598401913964, 'val_recall_macro': 0.7990187527864894, 'val_f1_macro': 0.6008038955458842, 'val_loss': 0.6932190395723201, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 559, np.int64(2): 123, np.int64(4): 91, np.int64(0): 78, np.int64(1): 74, np.int64(3): 43, np.int64(6): 34})
Saved best EfficientNetV2 classifier. val_f1_macro=0.6008

Epoch 2/5


Validating EfficientNetV2: 100%|██████████| 126/126 [00:25<00:00,  5.02it/s]


{'epoch': 2, 'train_accuracy': 0.7481415732830357, 'train_balanced_accuracy': np.float64(0.7479503399812941), 'train_precision_macro': 0.5768289734817132, 'train_recall_macro': 0.7479503399812941, 'train_f1_macro': 0.6406859834328926, 'train_loss': 0.7321007664732964, 'val_accuracy': 0.783433133732535, 'val_balanced_accuracy': np.float64(0.803911290217944), 'val_precision_macro': 0.5945628273546033, 'val_recall_macro': 0.803911290217944, 'val_f1_macro': 0.6515137896590943, 'val_loss': 0.6398541187333745, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 599, np.int64(2): 109, np.int64(4): 104, np.int64(1): 76, np.int64(3): 49, np.int64(0): 39, np.int64(6): 26})
Saved best EfficientNetV2 classifier. val_f1_macro=0.6515

Epoch 3/5


...

{'epoch': 3, 'train_accuracy': 0.7756573837789859, 'train_balanced_accuracy': np.float64(0.7925621111554568), 'train_precision_macro': 0.6231230768145125, 'train_recall_macro': 0.7925621111554568, 'train_f1_macro': 0.6888465173847657, 'train_loss': 0.647771138952057, 'val_accuracy': 0.8243512974051896, 'val_balanced_accuracy': np.float64(0.8395298152289473), 'val_precision_macro': 0.7035198085055592, 'val_recall_macro': 0.8395298152289473, 'val_f1_macro': 0.7593356407904588, 'val_loss': 0.5504567567221889, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 620, np.int64(2): 133, np.int64(4): 99, np.int64(1): 64, np.int64(0): 52, np.int64(6): 19, np.int64(3): 15})
Saved best EfficientNetV2 classifier. val_f1_macro=0.7593

Epoch 4/5


Validating EfficientNetV2: 100%|██████████| 126/126 [00:25<00:00,  4.93it/s]


{'epoch': 4, 'train_accuracy': 0.7920781093975369, 'train_balanced_accuracy': np.float64(0.8025573257968998), 'train_precision_macro': 0.6526205617550981, 'train_recall_macro': 0.8025573257968998, 'train_f1_macro': 0.7133953397507486, 'train_loss': 0.5854772875348141, 'val_accuracy': 0.7764471057884231, 'val_balanced_accuracy': np.float64(0.8093992057692292), 'val_precision_macro': 0.6490799209915659, 'val_recall_macro': 0.8093992057692292, 'val_f1_macro': 0.6987291922110531, 'val_loss': 0.700830291401541, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 542, np.int64(4): 161, np.int64(1): 112, np.int64(2): 100, np.int64(0): 55, np.int64(3): 19, np.int64(6): 13})
No improvement: 1/7

Epoch 5/5


Validating EfficientNetV2: 100%|██████████| 126/126 [00:27<00:00,  4.64it/s]

{'epoch': 5, 'train_accuracy': 0.8045046044602241, 'train_balanced_accuracy': np.float64(0.828792417422913), 'train_precision_macro': 0.6896385244179098, 'train_recall_macro': 0.828792417422913, 'train_f1_macro': 0.7477310247736026, 'train_loss': 0.5252969363636332, 'val_accuracy': 0.810379241516966, 'val_balanced_accuracy': np.float64(0.8141937735339031), 'val_precision_macro': 0.6782397355213661, 'val_recall_macro': 0.8141937735339031, 'val_f1_macro': 0.7247201686177368, 'val_loss': 0.6081719516280168, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 590, np.int64(4): 134, np.int64(2): 100, np.int64(0): 78, np.int64(1): 70, np.int64(3): 15, np.int64(6): 15})
No improvement: 2/7
